# Feature engineering

This notebook converts the preprocessed Sentinel-5P XCH₄ dataset into a tabular forecasting dataset. It constructs the one-week-ahead target, lagged and rolling features, and cyclical seasonal features. Rows without a complete forecasting history are then removed, and the resulting model-ready dataset is saved in Parquet format for subsequent model training and evaluation.  

## 1. Feature engineering and target columns isolation

In [ ]:
import pandas as pd

from po_valley_methane_forecasting.paths import find_project_root
from po_valley_methane_forecasting.features import(
    construct_df,
    add_target_columns,
    create_analytical_features,
    create_statistical_features,
    encode_time_of_the_year,
    build_model_ready_dataframe
)

from po_valley_methane_forecasting.data_access import(
    load_xarray_dataset,
    save_feature_table
)

In [33]:
project_root=find_project_root()

interim_data_dir = project_root / "data" / "interim"
interim_data_dir.mkdir(parents=True, exist_ok=True)

candidate_path = interim_data_dir / "sentinel5p_ch4_po_valley_candidate_2019_2024.nc"

candidate_ds = load_xarray_dataset(candidate_path)

In [34]:
ch4_df= construct_df(candidate_ds)

In [4]:
print("Rows:", len(ch4_df))
print("Cells:", ch4_df["cell_id"].nunique())
print("Weeks:", ch4_df["week"].nunique())

assert ch4_df["cell_id"].nunique() == 2620
assert ch4_df["week"].nunique() == 314
assert len(ch4_df) == 2620 * 314

Rows: 822680
Cells: 2620
Weeks: 314


In [5]:
ch4_df=add_target_columns(ch4_df)

In [18]:
valid_forecast_horizons = (
    ch4_df["forecast_horizon_days"]
    .dropna()
)

print(
    valid_forecast_horizons.value_counts()
)

assert valid_forecast_horizons.eq(7).all()

forecast_horizon_days
7.0    820060
Name: count, dtype: int64


In [19]:
assert (
    ch4_df["forecast_horizon_days"]
    .isna()
    .sum()
    == ch4_df["cell_id"].nunique()
)

In [7]:
ch4_df=create_analytical_features(ch4_df)

In [8]:
ch4_df=create_statistical_features(ch4_df)

In [9]:
ch4_df=encode_time_of_the_year(ch4_df)

In [10]:
model_df=build_model_ready_dataframe(ch4_df)

## 2. Exploration of the processed dataframe

In [11]:
print("Original rows:", len(ch4_df))
print("Model-ready rows:", len(model_df))

retention_rate = (
    len(model_df) / len(ch4_df)
)

print(
    f"Retention rate: "
    f"{retention_rate:.2%}"
)

print(
    "Model-ready cells:",
    model_df["cell_id"].nunique(),
)

print(
    "Model-ready weeks:",
    model_df["week"].nunique(),
)

print(
    "Period:",
    model_df["week"].min(),
    "→",
    model_df["week"].max(),
)

Original rows: 822680
Model-ready rows: 175411
Retention rate: 21.32%
Model-ready cells: 2615
Model-ready weeks: 270
Period: 2019-01-13 00:00:00 → 2024-12-22 00:00:00


In [12]:
example_cell = model_df["cell_id"].iloc[0]

gap_check = ch4_df.loc[
    (ch4_df["cell_id"] == example_cell)
    & (
        ch4_df["week"].between(
            "2022-07-10",
            "2022-09-11",
        )
    ),
    [
        "week",
        "CH4",
        "ch4_prev_1w",
        "ch4_prev_2w",
        "ch4_mean_last_3w",
        "target_next_week",
    ],
]

gap_check

,week,CH4,ch4_prev_1w,ch4_prev_2w,ch4_mean_last_3w,target_next_week
184,2022-07-10,1889.895386,1883.860107,NaN,NaN,1892.848633
185,2022-07-17,1892.848633,1889.895386,1883.860107,1888.868042,NaN
186,2022-07-24,NaN,1892.848633,1889.895386,NaN,NaN
187,2022-07-31,NaN,NaN,1892.848633,NaN,NaN
188,2022-08-07,NaN,NaN,NaN,NaN,NaN
189,2022-08-14,NaN,NaN,NaN,NaN,1898.372314
190,2022-08-21,1898.372314,NaN,NaN,NaN,1913.169800
191,2022-08-28,1913.169800,1898.372314,NaN,NaN,1883.055786
192,2022-09-04,1883.055786,1913.169800,1898.372314,1898.199300,1897.853271
193,2022-09-11,1897.853271,1883.055786,1913.169800,1898.026286,1895.870972


The long data gap in summer 2022 provides a useful sanity check: missing weeks must propagate through lagged and rolling features rather than being treated as consecutive observations

In [14]:
number_of_candidate_cells = int(
    candidate_ds["candidate_mask"].sum().values
)

number_of_candidate_cells

2620

In [20]:
assert len(model_df) > 0

assert model_df.notna().all().all()

assert model_df["cell_id"].nunique() <= number_of_candidate_cells

assert (
    model_df["target_week"] - model_df["week"]
).dt.days.eq(7).all()

print("Final dataset checks passed.")

Final dataset checks passed.


## 3. Saving dataset for machine learning

In [27]:
columns_to_save = [
    "week",
    "target_week",
    "cell_id",
    "x",
    "y",
    "CH4",
    "ch4_prev_1w",
    "ch4_prev_2w",
    "ch4_change_1w",
    "ch4_mean_last_3w",
    "ch4_std_last_3w",
    "season_sin",
    "season_cos",
    "target_next_week",
]

model_df_to_save = (
    model_df[columns_to_save]
    .sort_values(["week", "cell_id"])
    .reset_index(drop=True)
)

In [28]:
processed_data_dir = (
    project_root
    / "data"
    / "processed"
)

multiyear_features_path = (
    processed_data_dir
    / "methane_forecasting_features_2019_2024.parquet"
)

save_feature_table(model_df_to_save, multiyear_features_path)

print("Saved to:", multiyear_features_path)

Saved to: c:\Users\Pietro\OneDrive\Desktop\po_valley_methane_forecasting\data\processed\methane_forecasting_features_2019_2024.parquet


In [30]:
saved_model_df = pd.read_parquet(
    multiyear_features_path
)

print("Saved shape:", saved_model_df.shape)
print("Saved period:")
print(
    saved_model_df["week"].min(),
    "→",
    saved_model_df["week"].max(),
)

print("Saved cells:", saved_model_df["cell_id"].nunique())
print("Saved weeks:", saved_model_df["week"].nunique())

assert saved_model_df.shape == model_df_to_save.shape
assert list(saved_model_df.columns) == list(
    model_df_to_save.columns
)
assert saved_model_df.notna().all().all()

print("Saved Parquet dataset successfully verified.")

Saved shape: (175411, 14)
Saved period:
2019-01-13 00:00:00 → 2024-12-22 00:00:00
Saved cells: 2615
Saved weeks: 270
Saved Parquet dataset successfully verified.
